# Tesla Finanzanalyse #

## Installation der benötigten Python-Bibliotheken ##

In [1]:
pip install yfinance pandas matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


## Import der Analyse- & Visualisierungs-Frameworks ##

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
from datetime import datetime
from requests.exceptions import RequestException

## Datenimport über Yahoo Finance ##


In [3]:
def get_stock_data(ticker_symbol, start, end):
    """
    Lädt Finanzdaten einer Aktie von Yahoo Finance.

    Parameter:
    -----------
    ticker_symbol : str
        Das eindeutige Börsenkürzel der Aktie (z.B. "AAPL", "META").
    start : str
        Das Startdatum des Analysezeitraums im Format "YYYY-MM-DD".
    end : str
        Das Enddatum des Analysezeitraums im Format "YYYY-MM-DD".
        
    """
    print(f"Lade Daten für {ticker_symbol} von {start} bis {end}...")
    
    try:
        # Daten mit yfinance herunterladen
        stock_data = yf.download(tickers=ticker_symbol, start=start, end=end)
        
        # Prüfen, ob die Tabelle leer ist (z.B. wegen Tippfehler im Ticker)
        if stock_data.empty:
            raise ValueError(f"Keine Daten gefunden für das Kürzel '{ticker_symbol}'. Bitte überprüfe die Eingabe.")
   
    except ValueError as val_err:
        # Hier wird die eigens geworfene Exception bei leeren Daten abgefangen
        print(f"Validierungsfehler: {val_err}")
        return None
        
    except Exception as e:
        # Hier werden alle unerwarteten, technischen Fehler abgefangen (z.B. kein Internet)
        print(f"Technischer Fehler beim Datenabruf: {e}")
        return None

    return stock_data

## Bereinigung & Transformation der historischen Kursdaten ##

In [4]:
def clean_stock_data(stock_data):
    """
    Bereinigt die von yfinance ausgegebenen Daten.

    1. Entfernt alle Zeilen mit fehlenden Werten (NaNs).
    2. Setzt den Index zurück, um das "Date" (Datum) als reguläre Spalte zu nutzen.
    3. Normalisiert die 'Date'-Spalte, indem zeitzonenabhängige Informationen 
       (z.B. UTC) entfernt werden, um Formatkonflikte beim Plotten und in 
       der Statistik zu verhindern.
    4. Flacht Multi-Index-Spalten ab (Tupel, die von yfinance generiert werden),
       um sie in einfache Strings umzuwandeln (z.B. wird ("Close", "TSLA") zu "Close").
    5. Filtert das DataFrame, sodass nur die essenziellen Finanzspalten einbezogen werden.
    
    Parameter:
    -----------
    stock_data: pd.DataFrame
    
    """

    # Daten bereinigen, indem fehlende Werte gestrichen werden
    stock_data = stock_data.dropna()
        
    # Index zurücksetzen, um das Datum als Spalte einzubeziehen
    stock_data.reset_index(inplace=True)
        
    # Daten-Normalisierung der Zeitreihe: Falls Spalte 'Date' existiert & als Datetime-Objekt formatiert ist, wird eine evtl. vorhandene Zeitzone entfernt
    if "Date" in stock_data.columns and pd.api.types.is_datetime64_any_dtype(stock_data["Date"]) and stock_data["Date"].dt.tz is not None:
        stock_data["Date"] = stock_data["Date"].dt.tz_localize(None)

    # Die Spaltenstruktur anpassen (da die Spaltenstruktur von yfinance als Tupel übernommen wird); Macht aus der Spalte ("Close", "TSLA") einfach "Close"
    stock_data.columns = [col[0] if isinstance(col, tuple) else col for col in stock_data.columns]
        
    # Nur die Spalten behalten, die benötigt werden
    required_columns = ["Date", "Open", "High", "Low", "Close", "Volume"]
    existing_columns = [col for col in required_columns if col in stock_data.columns]
        
    print("\nDaten erfolgreich geladen und bereinigt!\n")
    return stock_data[existing_columns]

## Berechnung der finanzmathematischen Basiskennzahlen (Renditen & Kursveränderung) ##

In [5]:
def calculate_metrics(stock_data):
    """
    Berechnet grundlegende finanzmathematische Kennzahlen für die jeweilige Zeitreihe.

    Diese Funktion erweitert das übergebene DataFrame um die täglichen Renditen 
    sowie die absoluten Kursveränderungen zum Vortag (Gewinn/Verlust). Zudem wird vorab geprüft, 
    ob alle dafür notwendigen Kursspalten vorhanden sind.

    Parameter:
    -----------
    stock_data: pd.DataFrame
    
    """
    print("\nBerechne tägliche Renditen und absolute Gewinne/Verluste...")
    
    try:
        required_cols = ["Close", "High", "Low", "Open"]
        if not all(col in stock_data.columns for col in required_cols):
            raise KeyError("Erforderliche Spalten ('Close', 'High', 'Low' oder 'Open') fehlen in den Daten.")
            
        # Berechnung der täglichen Rendite über die 'Close'-Spalte (täglicher Schlusskurs): .pct_change() 
        # Rendite = (Abschlusskurs Heute - Abschlusskurs Gestern) geteilt durch Abschlusskurs Gestern
        stock_data["Daily_Return"] = stock_data["Close"].pct_change()

        # Berechnung der absoluten Kursveränderung (Gewinn/Verlust) in USD zum Vortag: .diff()
        # absolute Differenz des Schlusskurses zum vorherigen Handelstag: Schlusskurs Heute - Schlusskurs Gestern
        stock_data["Daily_Gain_Loss_USD"] = stock_data["Close"].diff()

        print("\nRenditen und innertägige Kennzahlen erfolgreich berechnet!")
        return stock_data
        
    except KeyError as key_err:
        print(f" Fehler in den Datenstrukturen: {key_err}")
        return None
        
    except TypeError as type_err:
        print(f" Datentyp-Fehler bei der Berechnung: {type_err}")
        return None
        

## Berechnung fortgeschrittener Risiko- und Trendmetriken ##

In [6]:
def calculate_advanced_metrics(stock_data):
    """
    Berechnet fortgeschrittene Risiko- und Trendmetriken für die jeweilige Zeitreihe.

    Diese Funktion erweitert das übergebene DataFrame um statistische Maße zur 
    Risikobeurteilung (rollierende historische Volatilität und pfadabhängige Drawdowns) 
    sowie um gleitende Durchschnitte zur technischen Trendanalyse (SMA 50 & SMA 200).

    Parameter
    ----------
    stock_data : pd.DataFrame
    
    """
    print("\nStarte erweiterte Finanzanalyse...")

   # Prüfung, ob alle Basis-Spalten aus den Phasen davor vorhanden sind
    try:
        required_advanced_cols = ["Daily_Return", "Close", "Volume", "High", "Low", "Open"]
        if not all(col in stock_data.columns for col in required_advanced_cols):
            raise KeyError("Erforderliche Spalten für die erweiterte Analyse fehlen.")

        # Berechnen der Drawdown-Zeitreihe (für die Kennzahlen)
        rolling_peak = stock_data["Close"].cummax()
        stock_data["Drawdown"] = (stock_data["Close"] - rolling_peak) / rolling_peak
    
        # Der finale globale Kennwert
        max_drawdown_global = stock_data["Drawdown"].min()
        print(f"Maximum Drawdown erfolgreich berechnet: {max_drawdown_global * 100:.2f}%")

        # Berechnung der rollierenden 21-Tage-Volatilität (Annualisierte 21-Tage-Standardabweichung der täglichen Renditen)
        # (Standardabweichung * Wurzel aus 252 zur Annualisierung; 21 Handelstage entsprechen ca. einem Kalendermonat) 
        # .std() berechnet die Standardabweichung im gleitenden Fenster, np.sqrt(252) rechnet es aufs Jahr hoch
        stock_data["Rolling_Volatility_21d"] = stock_data["Daily_Return"].rolling(window=21).std() * np.sqrt(252)

        # Trends: Berechnung des gleitenden Durchschnitts für 50-Handelstage und 200-Handelstage (SMA = simple moving averages)
        stock_data["SMA_50"] = stock_data["Close"].rolling(window=50).mean()
        stock_data["SMA_200"] = stock_data["Close"].rolling(window=200).mean()

        print("\nErweiterte Finanzanalyse erfolgreich abgeschlossen!")
        return stock_data

    except Exception as e:
        print(f"Fehler bei der erweiterten Finanzanalyse: {e}")
        return None

## Implementierung der Visualisierungsfunktionen ##

### Visualisierung des historischen Kursverlaufs (anhand der Schlusskurse) ###

In [7]:
def plot_price_history(stock_data, ticker_symbol):
    """
    Erstellt und speichert ein Liniendiagramm des historischen Schlusskursverlaufs.

    Diese Funktion visualisiert die zeitliche Entwicklung des Schlusskurses (Close) 
    einer Aktie und wendet ein standardisiertes Layout- und Chart-Styling an.
    Das resultierende Diagramm wird als PNG-Datei exportiert.

    Parameter
    ----------
    stock_data : pd.DataFrame
    ticker_symbol : str
    
    """
    print(f"\nGeneriere reinen Kursverlauf-Plot für {ticker_symbol}...")

    # Initialisierung des Diagramms mit vordefiniertem Stil und Abmessungen
    plt.style.use("seaborn-v0_8-whitegrid")
    plt.figure(figsize=(12, 6))

    # Generierung des Liniencharts für die Schlusskurse
    plt.plot(stock_data["Date"], stock_data["Close"], color="#1f77b4", linewidth=1.5, label="Schlusskurs (Close)")
    
    # Anwendung des standardisierten Chart-Stylings
    apply_standard_chart_styling(title=f"Historischer Kursverlauf ({ticker_symbol})", ylabel="Kurs in USD", show_grid=True)
    
    # Export des Diagramms & Schließen
    plt.savefig(f"{ticker_symbol}_01_kursverlauf.png", dpi=300)
    plt.close()
    print("Kursverlauf-Diagramm separat gespeichert.")

### Visualisierung der täglichen Renditedynamik ###

In [8]:
def plot_daily_returns(stock_data, ticker_symbol):
    """
    Erstellt und speichert ein Liniendiagramm der täglichen prozentualen Renditen:

    Diese Funktion visualisiert die zeitliche Entwicklung der Renditen, um Phasen 
    erhöhter Volatilität sichtbar zu machen. Das Diagramm wird zur 
    besseren Orientierung um eine Nulllinie ergänzt und als PNG-Datei exportiert.

    Parameter
    ----------
    stock_data : pd.DataFrame
    ticker_symbol : str
    
    """
    print(f"\nGeneriere Renditedynamik-Plot für {ticker_symbol}...")

    # Initialisierung des Diagramms mit vordefiniertem Stil und Abmessungen
    plt.style.use("seaborn-v0_8-whitegrid")
    plt.figure(figsize=(12, 6))
    
    # Umrechnung der Renditen in Prozent (für die Y-Achse)
    daily_return_pct = stock_data["Daily_Return"] * 100
    
    # Generierung des Liniencharts mit geringer Linienstärke für eine präzise Darstellung der Spikes
    plt.plot(stock_data["Date"], daily_return_pct, color="#ff7f0e", alpha=0.6, linewidth=0.7, label="Tägliche Rendite")
    
    # Implementierung einer horizontalen roten Nulllinie zur Orientierung
    plt.axhline(0, color="red", linestyle="--", linewidth=0.8, alpha=0.6)

    # Anwendung des standardisierten Chart-Stylings
    apply_standard_chart_styling(title=f"Tägliche Renditedynamik ({ticker_symbol})", ylabel="Rendite in %", show_grid=True)

    # Export des Diagramms & Schließen
    plt.savefig(f"{ticker_symbol}_02_taegliche_renditen.png", dpi=300)
    plt.close()
    print("Renditen-Diagramm separat gespeichert.")



def plot_returns_histogram(stock_data, ticker_symbol):
    """
    Erstellt und speichert ein separates Histogramm der täglichen prozentualen Renditen,
    um die statistische Häufigkeitsverteilung (Risikostruktur) aufzuzeigen.

    Parameter
    ----------
    stock_data : pd.DataFrame
    ticker_symbol : str
    """
    print(f"\nGeneriere Rendite-Histogramm für {ticker_symbol}...")

    # Initialisierung des Diagramms mit vordefiniertem Stil und Abmessungen
    plt.style.use("seaborn-v0_8-whitegrid")
    plt.figure(figsize=(12, 6))
    
    # Umrechnung der Renditen in Prozent
    daily_return_pct = stock_data["Daily_Return"] * 100

    # Histogramm mit der geglätteten Dichtekurve plotten
    sns.histplot(daily_return_pct, bins=80, kde=True, color="#ff7f0e", alpha=0.5, label="Häufigkeit")
    
    # Vertikale rote Null-Linie zur Orientierung (analog zur horizontalen Linie)
    plt.axvline(0, color="red", linestyle="--", linewidth=0.8, alpha=0.6)

    # Anwendung des standardisierten Chart-Stylings
    apply_standard_chart_styling(title=f"Häufigkeitsverteilung der täglichen Renditen ({ticker_symbol})", ylabel="Anzahl Tage", xlabel="Rendite in %", show_grid=True)
    
    # Export des Diagramms & Schließen
    plt.savefig(f"{ticker_symbol}_03_renditen_histogramm.png", dpi=300)
    plt.close()
    print("Renditen-Histogramm separat gespeichert.")

### Visualisierung des Kursverlaufs mit Trendindikatoren ###

In [9]:
def plot_price_trends(stock_data, ticker_symbol):
    """
    Erstellt und speichert ein Liniendiagramm des Schlusskurses inklusive SMA 50 und SMA 200 (50- & 20-Tage-Trend)

    Diese Funktion überlagert den historischen Schlusskursverlauf mit zwei gleitenden 
    Durchschnitten (SMA = simple moving averages), um mittelfristige und langfristige Trendwenden 
    (durch u.a. Schnittpunkte) visuell zu analysieren. Das fertige Diagramm wird als PNG-Datei exportiert.

    Parameter
    ----------
    stock_data : pd.DataFrame
    ticker_symbol : str
    
    """
    print(f"\nGeneriere Kurs- und Trend-Diagramm für {ticker_symbol}...")
    
    # Initialisierung des Diagramms mit vordefiniertem Stil und Abmessungen
    plt.style.use("seaborn-v0_8-whitegrid")
    plt.figure(figsize=(12, 6))
    
    # Visualisierung des Schlusskursverlaufs als Hauptlinie
    plt.plot(stock_data["Date"], stock_data["Close"], label="Schlusskurs (Close)", color="#1f77b4", linewidth=1.5)
    
    # Integration der Trendlinien (SMA 50 & SMA 200; SMA 50 als mittelfristiger Indikator & SMA 200 als langfristiger Indikator)
    plt.plot(stock_data["Date"], stock_data["SMA_50"], label="50-Tage-Trend (SMA 50)", color="#ff7f0e", linestyle="--", linewidth=1.2)
    plt.plot(stock_data["Date"], stock_data["SMA_200"], label="200-Tage-Trend (SMA 200)", color="#2ca02c", linestyle="-.", linewidth=1.2)

    # Automatische Berechnung der SMA-Schnittpunkte
    # Zeitpunkt, an dem die 50er-Linie über/unter der 200er-Linie liegt
    stock_data["Signal"] = stock_data["Rolling_Volatility_21d"]
    sma50 = stock_data["SMA_50"]
    sma200 = stock_data["SMA_200"]
    
    # Schnittpunkte mathematisch bestimmen (Wo ändert sich das Vorzeichen der Differenz?)
    idx_vorzeichenwechsel = (sma50 > sma200).diff()
    
    # Golden Cross (Schnittpunkt nach oben)
    golden_crosses = stock_data[idx_vorzeichenwechsel & (sma50 > sma200)]
    plt.scatter(golden_crosses["Date"], golden_crosses["SMA_50"], 
                color="green", marker="^", s=100, zorder=5, label="Golden Cross (Kaufsignal)")
    
    # Death Cross (Schnittpunkt nach unten)
    death_crosses = stock_data[idx_vorzeichenwechsel & (sma50 < sma200)]
    plt.scatter(death_crosses["Date"], death_crosses["SMA_50"], 
                color="red", marker="v", s=100, zorder=5, label="Death Cross (Verkaufsignal)")

    # Anwendung des standardisierten Chart-Stylings
    apply_standard_chart_styling(title=f"Kursverlauf und Trendanalysen ({ticker_symbol})", ylabel="Kurs in USD")
    
    # Export des Diagramms & Schließen
    plt.savefig(f"{ticker_symbol}_kurs_trends.png", dpi=300)
    plt.close()
    print("Trend-Diagramm als '..._kurs_trends.png' gespeichert.")

### Visualisierung der Risikometriken ###

In [10]:
def plot_risk_analysis(stock_data, ticker_symbol):

    """
    Erstellt und speichert ein Liniendiagramm der rollierenden 21-Tage-Volatilität,
    um das dynamische Marktrisiko im Zeitverlauf darzustellen.

    Diese Funktion visualisiert die historische, annualisierte Volatilität (mittelfristiges
    Marktrisiko) als Trendlinie, um Phasen erhöhter Marktunsicherheit und
    Regimewechsel in der Risikostruktur der Aktie zu identifizieren.
    Das Diagramm wird als PNG-Datei exportiert.

    Parameter
    ----------
    stock_data : pd.DataFrame
    ticker_symbol : str
       
    """
    print(f"\nGeneriere Risiko-Diagramm für {ticker_symbol}...")

    # Initialisierung des Diagramms mit vordefiniertem Stil und Abmessungen
    plt.style.use("seaborn-v0_8-whitegrid")
    plt.figure(figsize=(12, 6))
    
    # Umrechnung der Volatilitätswerte in Prozentwerte
    vola_21d_pct = stock_data["Rolling_Volatility_21d"] * 100
    
   # Plot der historischen Volatilität
    plt.plot(stock_data["Date"], vola_21d_pct, label="21-Tage-Volatilität (annualisiert)", color="#d62728", linewidth=1.8)

    # Anwendung des standardisierten Chart-Stylings
    apply_standard_chart_styling(title=f"Historischer Risikoverlauf ({ticker_symbol}) - Mittelfristiges Marktrisiko", xlabel="Datum", ylabel="Annualisierte Schwankungsbreite in %", show_grid=True, legend_loc="upper right")
    
    # Export des Diagramms & Schließen
    plt.tight_layout() # Verhindert das Zusammenquetschen der beiden Boxen
    plt.savefig(f"{ticker_symbol}_risikoanalyse.png", dpi=300)
    plt.close()
    print("Risiko-Diagramm als '..._risikoanalyse.png' gespeichert.")

### Visualisierung des Maximum Drawdown ###

In [11]:
def plot_drawdowns(stock_data, ticker_symbol):
    """
    Erstellt und speichert ein Liniendiagramm des historischen Drawdown-Verlaufs
    (Underwater-Chart), um Verlustphasen und Erholungszeiten aufzuzeigen.

    Diese Funktion visualisiert die pfadabhängige Verlustentwicklung der Aktie,
    indem der prozentuale Rückgang vom jeweils historisch gleitenden Höchststand 
    (Peak) über die Zeitachse abgetragen wird. Zudem wird der maximale Verlust (Maximum Drawdown) 
    automatisch im Datensatz lokalisiert, im Diagramm als roter Datenpunkt 
    hervorgehoben und mit dem exakten Prozentwert beschriftet.

    Parameter
    ---------
    stock_data : pd.DataFrame 
    ticker_symbol : str
    
    """
    print(f"\nGeneriere Drawdown-Diagramm für {ticker_symbol}...")

    # Initialisierung des Diagramms mit vordefiniertem Stil und Abmessungen
    plt.style.use("seaborn-v0_8-whitegrid")
    plt.figure(figsize=(12, 6))
    
    # Umrechnung in Prozent für die Darstellung
    drawdown_pct = stock_data["Drawdown"] * 100

    # Plot der Drawdown-Linie
    plt.plot(stock_data["Date"], drawdown_pct, color="#1f77b4", linewidth=1.5, label="Drawdown vom Allzeithoch")
    
    # Die Fläche unter der Kurve blau färben
    plt.fill_between(stock_data["Date"], drawdown_pct, 0, color="#1f77b4", alpha=0.2)

    # Globalen Tiefpunkt (Maximum Drawdown) im Chart markieren
    min_dd_idx = stock_data["Drawdown"].idxmin()
    min_dd_date = stock_data["Date"].iloc[min_dd_idx]
    min_dd_val = drawdown_pct.iloc[min_dd_idx]
    
    # Roter Punkt am tiefsten Stand
    plt.scatter(min_dd_date, min_dd_val, color="red", s=50, zorder=5, label=f"Max Drawdown: {min_dd_val:.2f}%")

    # Anwendung des standardisierten Chart-Stylings
    apply_standard_chart_styling(title=f"Historische Verlustphasen (Drawdowns) für {ticker_symbol}", ylabel="Rückgang vom Peak in %", xlabel="Datum", show_grid=True, legend_loc="lower left")

    # Achsen-Limitierung, da ein Drawdown nie über 0% steigen kann
    plt.ylim(top=2) 

    # Export des Diagramms & Schließen
    plt.tight_layout()
    plt.savefig(f"{ticker_symbol}_04_drawdowns.png", dpi=300)
    plt.close()
    
    print(f"Drawdown-Diagramm erfolgreich als '..._04_drawdowns.png' gespeichert.")

### Zentralisierung des standardisierten Diagramm-Designs ###

In [12]:
def apply_standard_chart_styling(title, ylabel, legend_loc="upper left", show_grid=False, xlabel="Datum"):
    """
    Wendet ein einheitliches Design und Layout auf das aktuelle Matplotlib-Diagramm an.

    Diese Hilfsfunktion zentralisiert wiederkehrende Formatierungen wie Achsenbeschriftungen,
    Schriftgrößen, Legendenpositionierung und optionale Gitterlinien, um Code-Doppelungen
    innerhalb der verschiedenen Visualisierungsfunktionen zu verhindern.

    Parameter
    ----------
    title : str
    ylabel : str
    legend_loc : str, optional
    show_grid : bool, optional
    xlabel : str, optional
    
    """
    #Konfiguration des Diagrammtitels und der Achsenbeschriftungen
    plt.title(title, fontsize=13, fontweight="bold", pad=15)
    plt.xlabel(xlabel, fontsize=12)
    plt.ylabel(ylabel, fontsize=12)
    
    # Flexible Platzierung und Formatierung der Legende
    plt.legend(loc=legend_loc, frameon=True, facecolor="white", edgecolor="none")
    
    # Optionale Implementierung eines Hintergrundgitters (Grid)
    if show_grid:
        plt.grid(True, linestyle=":", alpha=0.6)

    # Automatische Optimierung der Layout-Abstände zur Vermeidung von Abschneidungen
    plt.tight_layout()

## Zentraler Programmablauf (Main-Funktion) ##
Steuerung des gesamten Analyseprozesses: Interaktive Benutzereingabe, sequenzieller Datenabruf, Datenbereinigung, Berechnung der Finanzmetriken sowie die automatisierte Generierung und Speicherung aller Visualisierungen.

In [13]:
def main():
    """
    Steuert den gesamten Analyseprozess der historischen Aktiendaten.

    Diese Hauptfunktion steuert die sequenzielle Ausführung des Prozesses.
    Sie erfasst die Benutzereingaben, startet den Datenabruf sowie die Bereinigung,
    berechnet die verschiedenen Finanzkennzahlen und speichert am Ende alle Diagramme automatisch ab.

    Parameters
    ----------
    None
    
    """
    # Initiierung der interaktiven Benutzereingaben
    ticker_symbol = input("Gib das Aktienkürzel ein (z.B. 'TSLA'): ").strip()
    start_date = input("Gib das Startdatum ein (YYYY-MM-DD, z.B. '2021-01-01'): ").strip()
    end_date = datetime.now().strftime("%Y-%m-%d")
    
    print(f"\nStarte Analyseprozess für {ticker_symbol}...")
    print("--------------------------------------------------")
    
    # Datenabruf über Yahoo Finance
    stock_data = get_stock_data(ticker_symbol=ticker_symbol, start=start_date, end=end_date)

    if stock_data is None:
        print("Prozess abgebrochen: Datenabruf fehlgeschlagen.")
        return
    
    # Bereinigung und Transformation der Rohdaten
    cleaned_stock_data = clean_stock_data(stock_data)

    if cleaned_stock_data is None:
        print("Prozess abgebrochen: Datenbereinigung fehlgeschlagen.")
        return

   # Validierung der Daten
    cleaned_stock_data.info()

    # Berechnung der mathematischen Basis- und erweiterten Metriken
    cleaned_stock_data = calculate_metrics(stock_data=cleaned_stock_data)
    cleaned_stock_data = calculate_advanced_metrics(stock_data=cleaned_stock_data)
    
    if cleaned_stock_data is None:
        print("Prozess abgebrochen: Die Renditenberechnung und/oder Risikoanalyse ist fehlgeschlagen.")
        return

    # Konfiguration der Anzeige-Optionen für die Konsole
    pd.set_option("display.max_rows", 15)
    pd.set_option("display.max_columns", None)

    # Ausgabe der aktuellsten Datensätze (die letzten 5 Zeilen der Zeitreihe)
    print("\n=== Vorschau der bereinigten Finanzdaten ===")
    print(cleaned_stock_data.tail(5).to_string(index=False))
    print("===========================================================\n")

    # Generierung und Ausgabe der deskriptiven statistischen Kennzahlen
    print("\n=== Deskriptive Statistik der Finanzdaten ===")
    stats_cols = ["Close", "Daily_Return", "Rolling_Volatility_21d"]
    print(cleaned_stock_data[stats_cols].describe().to_string())
    print("================================================\n")

    # Automatisierte Erstellung und Speicherung aller Analyse-Diagramme
    print("Starte die Generierung der Diagramme...")
    plot_price_trends(stock_data=cleaned_stock_data, ticker_symbol=ticker_symbol)
    plot_risk_analysis(stock_data=cleaned_stock_data, ticker_symbol=ticker_symbol)
    plot_drawdowns(stock_data=cleaned_stock_data, ticker_symbol=ticker_symbol)
    plot_price_history(stock_data=cleaned_stock_data, ticker_symbol=ticker_symbol)
    plot_daily_returns(stock_data=cleaned_stock_data, ticker_symbol=ticker_symbol)
    plot_returns_histogram (stock_data=cleaned_stock_data, ticker_symbol=ticker_symbol)

    # Datenexport in eine strukturierte CSV-Datei (ersetzt zudem Punkte im Ticker (z.B. bei BRK.B), um Fehler im Dateipfad zu verhindern)
    file_name = f"{ticker_symbol.replace(".", "-")}_analysiert_{end_date}.csv"
    cleaned_stock_data.to_csv(file_name, index=False, sep=";", decimal=",")
    print(f"\nDaten erfolgreich gespeichert: '{file_name}'")

main()

Gib das Aktienkürzel ein (z.B. 'TSLA'):  TSLA
Gib das Startdatum ein (YYYY-MM-DD, z.B. '2021-01-01'):  2021-01-01



Starte Analyseprozess für TSLA...
--------------------------------------------------
Lade Daten für TSLA von 2021-01-01 bis 2026-07-19...


[*********************100%***********************]  1 of 1 completed



Daten erfolgreich geladen und bereinigt!

<class 'pandas.DataFrame'>
RangeIndex: 1390 entries, 0 to 1389
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype        
---  ------  --------------  -----        
 0   Date    1390 non-null   datetime64[s]
 1   Open    1390 non-null   float64      
 2   High    1390 non-null   float64      
 3   Low     1390 non-null   float64      
 4   Close   1390 non-null   float64      
 5   Volume  1390 non-null   int64        
dtypes: datetime64[s](1), float64(4), int64(1)
memory usage: 65.3 KB

Berechne tägliche Renditen und absolute Gewinne/Verluste...

Renditen und innertägige Kennzahlen erfolgreich berechnet!

Starte erweiterte Finanzanalyse...
Maximum Drawdown erfolgreich berechnet: -73.63%

Erweiterte Finanzanalyse erfolgreich abgeschlossen!

=== Vorschau der bereinigten Finanzdaten ===
      Date       Open       High        Low      Close   Volume  Daily_Return  Daily_Gain_Loss_USD  Drawdown  Rolling_Volatility_21d     SMA_50  